In [19]:
import albumentations as A
import cv2
import numpy as np
from pathlib import Path
import os


In [ ]:
class FaceDataAugmentation:
    def __init__(self, output_dir='augmented_faces'):
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)
        
        # Определяем pipeline аугментации
        self.transform = A.Compose([
            # Геометрические трансформации (умеренные для лиц)
            A.HorizontalFlip(p=0.5),
            A.Rotate(limit=15, p=0.5),  # Небольшие повороты
            A.ShiftScaleRotate(
                shift_limit=0.1,
                scale_limit=0.15,
                rotate_limit=10,
                p=0.5
            ),
            
            # Изменения освещения
            A.RandomBrightnessContrast(
                brightness_limit=0.2,
                contrast_limit=0.2,
                p=0.5
            ),
            A.RandomGamma(gamma_limit=(80, 120), p=0.3),
            A.CLAHE(p=0.3),  # Улучшение контраста
            
            # Цветовые трансформации
            A.HueSaturationValue(
                hue_shift_limit=10,
                sat_shift_limit=20,
                val_shift_limit=10,
                p=0.3
            )
        ])
    
    def augment_image(self, image):
        augmented = self.transform(image=image)
        return augmented['image']
    
    def augment_dataset(self, input_dir, images_per_original=10):
        """
        Аугментировать весь датасет
        
        Args:
            input_dir: папка с оригинальными изображениями
            images_per_original: сколько аугментированных версий создать
        """
        input_path = Path(input_dir)
        
        for person_dir in input_path.iterdir():
            if not person_dir.is_dir():
                continue
                
            person_name = person_dir.name
            output_person_dir = Path(self.output_dir) / person_name
            output_person_dir.mkdir(exist_ok=True)
            
            for img_path in person_dir.glob('*.jpg'):
                # Загрузка изображения
                image = cv2.imread(str(img_path))
                image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                
                # Сохранение оригинала
                original_name = f"{img_path.stem}_original.jpg"
                cv2.imwrite(
                    str(output_person_dir / original_name),
                    cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
                )
                
                # Создание аугментированных версий
                for i in range(images_per_original):
                    aug_image = self.augment_image(image)
                    aug_name = f"{img_path.stem}_aug_{i:03d}.jpg"
                    cv2.imwrite(
                        str(output_person_dir / aug_name),
                        cv2.cvtColor(aug_image, cv2.COLOR_RGB2BGR)
                    )
            
            print(f"  Создано {len(list(output_person_dir.glob('*.jpg')))} изображений")

In [7]:
augmentor = FaceDataAugmentation(output_dir='dataset/augmented')
augmentor.augment_dataset(input_dir='dataset/original', images_per_original=24)

  Создано 300 изображений


In [14]:
pic = cv2.imread('dataset/original/Vlad/IMG_0818.jpg')

In [36]:
import face_recognition
import cv2

# Загрузите изображение
image_path = "dataset/original/Vlad/IMG_3634.jpg"  # Укажите путь к вашему изображению
image = face_recognition.load_image_file('dataset/original/Vlad/IMG_3635.jpg')

# Найдите все лица в изображении
face_locations = face_recognition.face_locations(image)

# Выведите количество найденных лиц
print(f"Найдено {len(face_locations)} лиц(а) в изображении.")

# Отобразите изображение с обнаруженными лицами
image_with_faces = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

for (top, right, bottom, left) in face_locations:
    # Нарисуйте прямоугольник вокруг каждого лица
    cv2.rectangle(image_with_faces, (left, top), (right, bottom), (0, 255, 0), 2)

# Сохраните изображение с обнаруженными лицами
output_path = "output_image.jpg"
cv2.imwrite(output_path, image_with_faces)
print(f"Изображение сохранено как {output_path}")

# Закройте все окна



Найдено 1 лиц(а) в изображении.
Изображение сохранено как output_image.jpg


In [32]:
import face_recognition

In [37]:
face_locations

[(64, 201, 219, 46)]